# Player!

This takes the neural network from the mimicry notebook and runs it in realtime connected to the Pyphonic plugin (running standalone or in the DAW).

In [1]:
from pyphonic.src.pyphonic import Pyphonic, midi_parser

In [2]:
import math

import torch
import torch.nn.functional as F
import torch.nn as nn

import numpy as np

# Define the neural network

This must be the same as in the training notebook.

Note that when you train a network on a certain block size (e.g. 512 samples), it must be exactly the same here and in the audio coming from the plugin/DAW, else the neural network will fail because the input's the wrong size. Potentially solveable with masking but I haven't done that.

In [111]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=50):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model, device='cuda')
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.lpe = pe.unsqueeze(0)
        self.lpe = nn.Parameter(self.lpe)

    def forward(self, x):
        x = x + self.lpe[:x.size(1), :]
        return self.dropout(x)

def actfn(x):
    # bipolar sigmoid
    return (1 - torch.exp(-x)) / (1 + torch.exp(-x))

class Net (torch.nn.Module):
    def __init__(self, block_size=512, num_prev_states=50):
        super(Net, self).__init__()
        
        self.block_size = block_size
        self.num_prev_states = num_prev_states
        
        self.l0 = nn.Linear(self.block_size, self.block_size, bias=False).cuda()
        self.l1 = nn.Linear(self.block_size, self.block_size, bias=False).cuda()
        
        self.pe = PositionalEncoding(self.block_size, dropout=0., max_len=self.num_prev_states).cuda()
        
        self.q_layer = nn.Linear(self.block_size, self.block_size, bias=True).cuda()
        self.k_layer = nn.Linear(self.block_size, self.block_size, bias=True).cuda()
        self.v_layer = nn.Linear(self.block_size, self.block_size, bias=True).cuda()
        
        self.attn = nn.MultiheadAttention(self.block_size, self.block_size, bias=True)
        self.attn_out = nn.Linear(self.block_size, self.block_size, bias=True).cuda()
        
        self.residual_layer = nn.Linear(self.block_size, self.block_size, bias=True).cuda()
        self.final_layer = nn.Linear(self.block_size, self.block_size, bias=True).cuda()
        self.attention_pool = nn.Linear(self.block_size, 1, bias=True).cuda()
        
    def init_hidden(self, bs):
        
        self.past = torch.zeros((bs, self.num_prev_states, self.block_size), device='cuda')
    
    def forward(self, x):
        bs = x.size(0)
        
        x = actfn(self.l0(x.transpose(2,1)))
        x = actfn(self.l1(x))
        
        orig_x = x.clone()
        
        self.past = torch.cat((self.past[:, 1:, :], x), dim=1)
        past = self.pe(self.past)
        
        q = actfn(self.q_layer(past))
        k = actfn(self.k_layer(past))
        v = actfn(self.v_layer(past))
        
        q = q.transpose(1, 0) # (SEQ_LEN, BS, EMBED_DIM)
        k = k.transpose(1, 0)
        v = v.transpose(1, 0)
        
        x, _ = self.attn(q, k, v)
        
        x = actfn(self.attn_out(x))
        x = x.transpose(1, 0)
        
        x = torch.matmul(F.softmax(self.attention_pool(x), dim=1).transpose(-1, -2), x).squeeze(-2)
        
        x = x.unsqueeze(-1)
        
        x = x + self.residual_layer(orig_x).transpose(2, 1)
        
        x = self.final_layer(x.transpose(2,1)).transpose(2,1) # removed actfn
        return x

In [120]:
class MyCallbackDeepLearning:
    def __init__(self, bpm=120.0, chunk=512):
        self.bpm = bpm
        self.chunk = chunk
        self.net = Net(num_prev_states=100) # This was set in the training notebook
        self.net.load_state_dict(torch.load("overdrive_reverb_net_1024_mono.bin"))
        self.net.cuda()
        self.net.eval()
        self.net.init_hidden(2) # batch size 2 because incoming audio is stereo
        self.noise = self.net(torch.zeros((2, chunk, 1), device='cuda')).detach()
        # What you don't capture (because of the way it's trained) is any interplay between the two stereo channels.
    def __call__(self, audio_data, midi_data, bpm=128, chunk_size=512):
        audio_data = torch.tensor(audio_data, dtype=torch.float32, requires_grad=False, device='cuda')
        audio_data = audio_data.view(2, -1, 1)[0].repeat((2, 1, 1))
        with torch.no_grad():
            reply = self.net(audio_data)
        reply = reply - self.noise
        reply = reply.view(2, -1)
        torch.clamp_(reply, -1, 1)
        reply = reply.cpu().numpy()
        return reply, midi_data

In [121]:
p = Pyphonic(MyCallbackDeepLearning)

In [122]:
p.connect() # Ensure the plugin is running somewhere

In [123]:
p.go() # At this point, sound is coming out :)

'Connection dropped'